### This notebook will focus on finding the best model to predict the severity predictor !

In [184]:
### Libraries we will be using
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt 
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, accuracy_score,recall_score,f1_score,roc_curve,roc_auc_score, precision_recall_curve,confusion_matrix,ConfusionMatrixDisplay,classification_report, mean_absolute_error, cohen_kappa_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb


In [185]:
## Data set we will be using 
data = pd.read_csv('../Datasets/pre-processedData.csv')
attack_data = data[data['is_malicious'] == 1]
attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)
attack_data.head(5)

/var/folders/bt/9_3kc80d4jq8sv8fv_c7d_fr0000gn/T/ipykernel_64288/3148040238.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attack_data.drop(columns=['is_malicious', 'Unnamed: 0'], inplace = True)


,duration,src_bytes,dst_bytes,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,...,flag_REJ,flag_RSTO,flag_RSTR,flag_S1,flag_S3,flag_SF,flag_SH,Severity_Score,bytes_ratio,total_bytes
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,...,0,0,0,0,0,0,0,2,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,...,1,0,0,0,0,0,0,2,0.0,0.0
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,...,0,0,0,0,0,0,0,2,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,...,0,0,0,0,0,0,0,2,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0.0,0,...,0,0,0,0,0,0,0,2,0.0,0.0


In [186]:
## Train test split of the data 
RANDOM_SEED = 42
X = attack_data.drop(columns=['Severity_Score'])
y = attack_data['Severity_Score']

## Splitting the data into train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=RANDOM_SEED)



---

### Model 1 : Logisitc Regression 

In [187]:
### Setting custom class weights based on the severity type 
class_weights = {1:1,2:3, 3:10} 
pipeline_steps = [('scaler', StandardScaler()),('logit', LogisticRegression(solver='lbfgs', class_weight = class_weights,C=1000,penalty='l2',max_iter = 1000))]
logit_pipeline = Pipeline(pipeline_steps)

logit_model = logit_pipeline.fit(X_train,y_train)

/opt/anaconda3/envs/Network_Intrusion/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [188]:
logit_training_pred = logit_model.predict(X_train)
logit_testing_pred = logit_model.predict(X_test)
logit_training_proba = logit_model.predict_proba(X_train)[:,1]
logit_testing_proba = logit_model.predict_proba(X_test)[:,1]


In [189]:
## Evaluating the model
logit_mae = mean_absolute_error(y_test, logit_testing_pred)
print(f"{logit_mae:.3f}")

# Cohen Kappa Score 
cks = cohen_kappa_score(y_test, logit_testing_pred)
print(f"{cks:.2f}")


high_severe_testing_data = y_test == 3
high_severe_predicted_test_data = logit_testing_pred == 3

### Confusion Matrix for the severe cases 

true_positives = ((y_test == 3)& (logit_testing_pred == 3)).sum()
false_positives = ((y_test != 3) & (logit_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (logit_testing_pred != 3)).sum()
true_negatives = ((y_test != 3) & (logit_testing_pred != 3)).sum()



### Metrics for calculation 
logit_rare_precision = true_positives/(true_positives+false_positives) 
logit_rare_recall = true_positives/(true_positives + false_negatives)
logit_f1_rare = 2*((logit_rare_precision * logit_rare_recall)/(logit_rare_precision + logit_rare_recall))
logit_rare_fpr = false_positives/(true_negatives+false_positives) ## Misclassified
logit_rare_fnr = false_negatives/(true_positives+false_negatives) ## False Alaram

print(logit_rare_precision, logit_rare_recall, logit_rare_fpr, logit_rare_fnr)




0.007
0.99
0.9756986634264885 0.9901356350184957 0.001483459427384661 0.009864364981504316


---
### Model 2: Linear SVM 



In [190]:
pipeline_steps = [('scaler', StandardScaler()), ('svm', LinearSVC(multi_class='ovr', penalty='l2', C = 0.1))]
svm_pipeline = Pipeline(pipeline_steps)

best_svm = svm_pipeline.fit(X_train, y_train)

In [191]:
svm_testing_pred = best_svm.predict(X_test)
svm_training_pred = best_svm.predict(X_train)

svm_testing_boundary = best_svm.decision_function(X_test)
svm_training_boundary = best_svm.decision_function(X_train)

In [192]:
### Metrics for evaluation 
svm_mae = mean_absolute_error(y_test, svm_testing_pred)

## Cohen Kappa Score 
svm_cohen_kappa = cohen_kappa_score(y_test, svm_testing_pred)

## Confusion Matrix 

true_positives = ((y_test == 3) & (svm_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (svm_testing_pred != 3)).sum()
false_negatives = ((y_test == 3) & (svm_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (svm_testing_pred == 3 )).sum()


### Precision
svm_precision_rare = true_positives /(true_positives + false_positives)

## Recall
svm_recall_rare = true_positives/(true_positives + false_negatives)

## F1_score 

svm_f1_rare = 2*((svm_precision_rare * svm_recall_rare)/(svm_precision_rare + svm_recall_rare))

### False Positive Rate

svm_fpr = false_positives / (false_positives + true_negatives) ## Misclassified
svm_fnr = false_negatives / (false_negatives + true_positives) ## False Alaram


print(svm_precision_rare, svm_recall_rare, svm_f1_rare, svm_fpr, svm_fnr)

0.9887359198998749 0.9741060419235512 0.9813664596273292 0.0006675567423230974 0.025893958076448828


---
### Model 3: Naive Bayes 

In [193]:
naive_model = GaussianNB(var_smoothing=1e-9)
naive_model.fit(X_train, y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [194]:
## Training and testing pred

naive_testing_pred = naive_model.predict(X_test)
naive_training_pred = naive_model.predict(X_train)

naive_testing_proba = naive_model.predict_proba(X_test)
naive_training_proba = naive_model.predict_proba(X_train)


In [195]:
## Metrics for evaluation 

naive_mae = mean_absolute_error(y_test, naive_testing_pred)

## Cohen-Kappa Score

naive_cohen = cohen_kappa_score(y_test, naive_testing_pred)

## Confusion Matrix 

true_positives = ((y_test == 3) & (naive_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (naive_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (naive_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (naive_testing_pred != 3)).sum()

### Precision, Recall, F1-Score, fpr, fnr 

naive_precision = true_positives / (true_positives + false_positives)
naive_recall = true_positives / (true_positives + false_negatives)
naive_f1_rare = 2*((naive_precision * naive_recall)/(naive_precision + naive_recall))
naive_fpr = false_positives /(false_positives + true_negatives) ## Misclassified Cases
naive_fnr = false_negatives /(false_negatives + true_positives) ## False Alarm 



print(naive_mae, naive_cohen, naive_precision, naive_recall, naive_fpr, naive_fnr)

0.07535157069894353 0.8457851351117285 0.8656174334140436 0.8816276202219482 0.008233199821984869 0.11837237977805179


---

### Model 4: Decision Tree Classifier 

In [196]:
decision_model = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_SEED, max_depth = 10, max_features=12, min_samples_leaf=75, min_samples_split=230)

decision_model.fit(X_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",230
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",75
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",12
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [197]:
decision_testing_pred = decision_model.predict(X_test)
decision_training_pred = decision_model.predict(X_train)

decision_testing_proba = decision_model.predict_proba(X_test)[:,1]
decision_training_proba = decision_model.predict_proba(X_train)[:,1]

In [198]:
### Metrics for the evaluation 

## Mean Absoulute Error 
decision_mae = mean_absolute_error(y_test, decision_testing_pred)

## Cohen Kappa's 


decision_cohen = cohen_kappa_score(y_test, decision_testing_pred)

## Confusion matrix 

true_positives = ((y_test ==3) & (decision_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (decision_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (decision_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (decision_testing_pred != 3)).sum()


## Precision, recall and f1 score 


decision_precision = true_positives / (true_positives + false_positives)
decision_recall = true_positives / (true_positives + false_negatives)
decision_fpr = false_positives  / (false_positives + true_negatives) ##Mis classification error 
decision_fnr = false_negatives / (false_negatives + true_positives) ## False Alarm
decision_f1 = 2 * ((decision_precision * decision_recall) / (decision_precision + decision_recall))


print(decision_precision, decision_recall, decision_fpr, decision_fnr, decision_f1)

0.9509433962264151 0.9321824907521579 0.002892745883400089 0.06781750924784218 0.9414694894146949


---
### Model 5: Random Forest Classifier

In [199]:
estimators = [100,200,300,400,500]
oob_scores = []
for n in estimators:
    random_model = RandomForestClassifier(oob_score = True , n_estimators= n, min_samples_leaf=70, min_samples_split=160, max_depth = 12, max_features=8)
    random_model.fit(X_train,y_train)
    oob_scores.append(random_model.oob_score_)

In [200]:
random_model = RandomForestClassifier(criterion='gini', n_estimators=200, oob_score=True, max_depth = 11, max_features= 10, min_samples_leaf=70, min_samples_split=200, random_state= RANDOM_SEED)
random_model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",11
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",70
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",10
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [201]:
random_testing_pred = random_model.predict(X_test)
random_training_pred = random_model.predict(X_train)

random_testing_proba = random_model.predict_proba(X_test)
random_training_proba = random_model.predict_proba(X_train)

In [202]:
### Metrics for evaluation 

random_mae = mean_absolute_error(y_test, random_testing_pred)

## Cohen Kappa
random_cohen = cohen_kappa_score(y_test, random_testing_pred)

## Confusion Matrix 

## Confusion matrix 

true_positives = ((y_test ==3) & (random_testing_pred == 3)).sum()
true_negatives = ((y_test != 3) & (random_testing_pred != 3)).sum()
false_positives = ((y_test != 3) & (random_testing_pred == 3)).sum()
false_negatives = ((y_test == 3) & (random_testing_pred != 3)).sum()


## Precision, recall and f1 score 


random_precision = true_positives / (true_positives + false_positives)
random_recall = true_positives / (true_positives + false_negatives)
random_fpr = false_positives  / (false_positives + true_negatives) ##Mis classification error 
random_fnr = false_negatives / (false_negatives + true_positives) ## False Alarm
random_f1 = 2 * ((decision_precision * decision_recall) / (decision_precision + decision_recall))


print(random_precision, random_recall, random_fpr, random_fnr, random_f1)

0.9961340206185567 0.9531442663378545 0.00022251891410769915 0.0468557336621455 0.9414694894146949


---
### Model 6: XGBoost

In [203]:

## Only for XGBModel :
attack_data['Severity_Score'] = attack_data['Severity_Score'].map({1:0,2:1,3:2})
X = attack_data.drop(columns=['Severity_Score'])
y = attack_data['Severity_Score']

## Splitting the data into train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=RANDOM_SEED)

xgb_model = xgb.XGBClassifier(learning_rate = 1, n_estimators = 43, max_depth = 6, colsample_bytree = 0.6,objective="multi:softmax", reg_lambda = 0.1, subsample = 0.7, min_child_weight = 2)

xgb_model.fit(X_train,y_train)

/var/folders/bt/9_3kc80d4jq8sv8fv_c7d_fr0000gn/T/ipykernel_64288/733552882.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  attack_data['Severity_Score'] = attack_data['Severity_Score'].map({1:0,2:1,3:2})


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.6
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=

In [204]:
## Training and Testing Score

xgb_training_score = xgb_model.predict(X_train)
xgb_testing_score = xgb_model.predict(X_test)

print((xgb_testing_score == 1).sum())

xgb_training_proba = xgb_model.predict_proba(X_train)
xgb_testing_proba = xgb_model.predict_proba(X_test)


10707


In [205]:
## Metrics of evaluation 

xgb_mae = mean_absolute_error(y_test, xgb_testing_score)


## Cohen Kappa's score

xgb_cohen = cohen_kappa_score(y_test, xgb_testing_score)


## Confusion matrix 

true_positives = ((y_test ==2) & (xgb_testing_score == 2)).sum()
true_negatives = ((y_test != 2) & (xgb_testing_score != 2)).sum()
false_positives = ((y_test != 2) & (xgb_testing_score == 2)).sum()
false_negatives = ((y_test == 2) & (xgb_testing_score != 2)).sum()


## Precision, recall and f1 score 


xgb_precision = true_positives / (true_positives + false_positives)
xgb_recall = true_positives / (true_positives + false_negatives)
xgb_fpr = false_positives  / (false_positives + true_negatives) ##Mis classification error 
xgb_fnr = false_negatives / (false_negatives + true_positives) ## False Alarm
xgb_f1 = 2 * ((xgb_precision * xgb_recall) / (xgb_precision +xgb_recall))


print(xgb_precision, xgb_recall, xgb_fpr, xgb_fnr, xgb_f1)

0.9975247524752475 0.9938347718865598 0.0001483459427384661 0.006165228113440197 0.9956763434218652


---
### Lets see how did every model perform in terms of severity classification ! 

In [208]:
model_name = ['LR', 'SVM', 'GaussianNB', 'DecisionTrees', 'RandomForest', 'XGBoost']

## Model's Mean-Absolute Error

model_mae = [logit_mae, svm_mae, naive_mae, decision_mae, random_mae, xgb_mae]

## Model's Cohen Kappa Score

model_cohen = [cks, svm_mae, naive_mae, decision_mae, random_mae, xgb_mae]

## Model's precision score

model_precision = [logit_rare_precision, svm_precision_rare, naive_precision, decision_precision, random_precision, xgb_precision]

## Model's Recall Score

model_recall = [logit_rare_recall, svm_recall_rare, naive_recall, decision_recall, random_recall, xgb_recall]

## Model's F1 Score

model_f1 = [logit_f1_rare, svm_f1_rare, naive_f1_rare, decision_f1,random_f1, xgb_f1]

## Misclassification 
misclassification_error = [logit_rare_fpr, svm_fpr,naive_fpr, decision_fpr, random_fpr, xgb_fpr]

## False Alarm

false_alarm = [logit_rare_fnr,svm_fnr,naive_fnr,decision_fnr, random_fnr, xgb_fnr]
severity_classifiers = pd.DataFrame(
{
    'model_name': model_name,
    'model_mae': model_mae,
    'model_cohen':model_cohen,
    'precision_rare_attacks':model_precision,
    'recall_rare_attacks': model_recall,
    'f1_rare_attacks': model_f1,
    'Misclassified_attacks':misclassification_error,
    'false_alarm':false_alarm
}
)

severity_classifiers.head(6)

,model_name,model_mae,model_cohen,precision_rare_attacks,recall_rare_attacks,f1_rare_attacks,Misclassified_attacks,false_alarm
0,LR,0.007416,0.986266,0.975699,0.990136,0.982864,0.001483,0.009864
1,SVM,0.007066,0.007066,0.988736,0.974106,0.981366,0.000668,0.025894
2,GaussianNB,0.075352,0.075352,0.865617,0.881628,0.873549,0.008233,0.118372
3,DecisionTrees,0.015252,0.015252,0.950943,0.932182,0.941469,0.002893,0.067818
4,RandomForest,0.006996,0.006996,0.996134,0.953144,0.941469,0.000223,0.046856
5,XGBoost,0.001189,0.001189,0.997525,0.993835,0.995676,0.000148,0.006165
